## DMRG WITH TENPY

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import pickle
import os
from datetime import datetime

from scipy.optimize import curve_fit

np.set_printoptions(precision=5, suppress=True, linewidth=100)
plt.rcParams['figure.dpi'] = 150
tenpy.tools.misc.setup_logging(to_stdout="INFO")

In [19]:
L = 10

filename_master = os.path.join(os.getcwd(), f"L{L}-full")

## Sanity Check


In [52]:
from tenpy import CouplingMPOModel, SpinHalfSite


class myModel(CouplingMPOModel):
    def init_sites(self, model_params):
        site = SpinHalfSite(conserve=None)
        return site
    
    def init_terms(self, model_params):
        Jx = model_params.get('Jx', 1.0)
        hz = model_params.get('hz', 0.0)
        kl = model_params.get('kl', 0.0)
        kr = model_params.get('kr', 0.0)
        truth = model_params.get('truth', True)

        # Onsite magnetic field
        for u in range(len(self.lat.unit_cell)):
            self.add_onsite(hz, u, 'Sigmaz')

        # Nearest-neighbor XX coupling
        for u1, u2, dx in self.lat.pairs['nearest_neighbors']:
            self.add_coupling(Jx, u1, 'Sigmax', u2, 'Sigmax', dx)

        # 3-site interactions (already Hermitian)
        dx = [(0,), (1,), (2,)]
        u = 0  # single-site unit cell

        self.add_multi_coupling(
            kl,
            [('Sigmax', dx[0], u), ('Sigmay', dx[1], u), ('Sigmaz', dx[2], u)]
        )
        self.add_multi_coupling(
            -kl,
            [('Sigmay', dx[0], u), ('Id', dx[1], u), ('Sigmax', dx[2], u)]
        )
        self.add_multi_coupling(
            -kr,
            [('Sigmaz', dx[0], u), ('Sigmay', dx[1], u), ('Sigmax', dx[2], u)]
        )
        self.add_multi_coupling(
            kr,
            [('Sigmax', dx[0], u), ('Id', dx[1], u), ('Sigmay', dx[2], u)]
        )

        # add boundary conditions
        if truth:
            self.add_onsite_term(10, 0, "Sigmax")
            self.add_onsite_term(-10, L-1, "Sigmax")

def dmrg_sim(Jx, hz, kl, kr, filename, truth = True):
    import tenpy
    import tenpy.linalg.np_conserved as npc
    from tenpy.algorithms import dmrg
    from tenpy.networks.mps import MPS
    from tenpy.models.tf_ising import TFIChain
    dmrg_params = {
    'mixer': True,  # setting this to True helps to escape local minima
    'max_E_err': 1.e-4,
    'trunc_params': {
        'chi_max': 100,
        'svd_min': 1.e-10,
    },
    'combine': True
    }

    model_params = {
    'Jx': Jx , 'hz': hz, 'kl': kl, 'kr': kr,
    'L': L, 'truth': truth,
    'bc_MPS': 'finite',
    }

    M = myModel(model_params)
    psi = MPS.from_lat_product_state(M.lat, [['up']])

    eng = dmrg.TwoSiteDMRGEngine(psi, M, dmrg_params)
    E, psi = eng.run() # the main work; modifies psi in place

    # save to file
    data = {'psi': psi, 'dmrg_params': dmrg_params, 'model_params': model_params}

    os.makedirs(filename, exist_ok=True)
        
    loc = os.path.join(filename, f'{hz:.3f}_{kl:.3f}.pkl')
    with open(loc, 'wb') as f:
        pickle.dump(data, f)
    
    return E, M

def dmrg_lines(start, stop, step, fixed, filename, setval="k", Jx=1):
    values = np.arange(start, stop+step, step)

    for sweep in values:
        if setval == "k":
            hz = sweep
            k = fixed
        elif setval == "h":
            hz = fixed
            k = sweep
        print(Jx, hz, k)
        _, M = dmrg_sim(Jx, hz, k, k, filename)
        print(M.all_coupling_terms().to_TermList() + M.all_onsite_terms().to_TermList())


In [54]:
E, M = dmrg_sim(1, 1, 0.5, 0.5, filename, truth=False)

INFO    : myModel: reading 'bc_MPS'='finite'
INFO    : myModel: reading 'L'=10
INFO    : myModel: reading 'Jx'=1
INFO    : myModel: reading 'hz'=1
INFO    : myModel: reading 'kl'=0.5
INFO    : myModel: reading 'kr'=0.5
INFO    : myModel: reading 'truth'=False
INFO    : TwoSiteDMRGEngine: subconfig 'trunc_params'=Config(<2 options>, 'trunc_params')
INFO    : TwoSiteDMRGEngine: reading 'combine'=True
INFO    : TwoSiteDMRGEngine: reading 'mixer'=True
INFO    : activate DensityMatrixMixer with initial amplitude 1e-05
INFO    : Running sweep with optimization
INFO    : trunc_params: reading 'chi_max'=100
INFO    : trunc_params: reading 'svd_min'=1e-10
INFO    : checkpoint after sweep 1
energy=-15.0931570066462211, max S=0.7579106008148944, age=10, norm_err=9.2e-02
Current memory usage 444.7MB, wall time: 4.5s
Delta E = nan, Delta S = 5.9448e-01 (per sweep)
max trunc_err = 1.3201e-20, max E_trunc = 2.8422e-14
chi: [2, 4, 8, 16, 30, 16, 8, 4, 2]
INFO    : Running sweep with optimization
INFO 

In [41]:
# run the same simulations with PIETROS code:

class MPO_main():
    Id = np.identity(2)
    X = np.array([[0, 1], [1, 0]])
    Y = np.array([[0, 0-1j], [0+1j, 0]])
    Z = np.array([[1, 0], [0, -1]])
    
    def __init__(self, h, k_left, k_right, J, pol=None, d=2):
        self.h = h
        self.k_left = k_left
        self.k_right = k_right
        self.J = J

        self.pol = pol
        self.d = d

    def Wl(self):
        Wleft = np.zeros((2, 2, 9),dtype='complex')
        Wleft[:, :, 0] = MPO_main.Id
        Wleft[:, :, 1] = MPO_main.X
        Wleft[:, :, 2] = MPO_main.Y
        Wleft[:, :, 3] = MPO_main.Z
        Wleft[:, :, 8] = self.h * MPO_main.Z

        if self.pol == 'tot':
            Wleft[:,:,8] += 10*MPO_main.X

        return Wleft
    
    def Wr(self):
        Wright = np.zeros((2, 2, 9),dtype='complex')
        Wright[:, :, 0] =  self.h * MPO_main.Z
        Wright[:, :, 1] =  self.J * MPO_main.X
        Wright[:, :, 4] =  self.k_right * MPO_main.Y
        Wright[:, :, 5] = -self.k_left * MPO_main.X
        Wright[:, :, 6] =  self.k_left * MPO_main.Z
        Wright[:, :, 7] = -self.k_right * MPO_main.X
        Wright[:, :, 8] =  MPO_main.Id

        if self.pol == 'tot':
            Wright[:,:,0] -= 10*MPO_main.X

        return Wright
    
    def mpo(self, p=None):
        MPO = np.zeros((2, 2, 9, 9),dtype='complex')

        # All interactions
        MPO[:,:,0, 0] =  MPO_main.Id
        MPO[:,:,0, 1] =  MPO_main.X
        MPO[:,:,0, 2] =  MPO_main.Y
        MPO[:,:,0, 3] =  MPO_main.Z
        MPO[:,:,0, 8] =  self.h * MPO_main.Z

        MPO[:,:,1, 4] =  MPO_main.Id
        MPO[:,:,1, 6] =  MPO_main.Y
        MPO[:,:,2, 5] =  MPO_main.Id
        MPO[:,:,3, 7] =  MPO_main.Y

        MPO[:,:,1, 8] =  self.J * MPO_main.X
        MPO[:,:,4, 8] =  self.k_right * MPO_main.Y
        MPO[:,:,5, 8] = -self.k_left * MPO_main.X
        MPO[:,:,6, 8] =  self.k_left * MPO_main.Z
        MPO[:,:,7, 8] = -self.k_right * MPO_main.X 
        MPO[:,:,8, 8] =  MPO_main.Id

        return MPO
 
def dmrg_main(L, par, pol, task_id = '0_0', J = 1):
    from dmrg.obs import observables
    from dmrg.MPS import MPS
    from dmrg.cont import CONT
    from dmrg.dmrg import dmrg

    start_time = datetime.now()

    # Base output folder for this task
    base_path = task_id
    # os.makedirs(base_path, exist_ok=True)

    path_out = os.path.join(base_path, 'OUT')
    os.makedirs(path_out, exist_ok=True)
    path_mps = os.path.join(base_path, 'MPS')
    os.makedirs(path_mps, exist_ok=True)
    path_cont = os.path.join(base_path, 'CONT')
    os.makedirs(path_cont, exist_ok=True)

    # Define the system size and bond dimension
    L = L  # check
    chi = 100

    h_x, k_l, k_r = par

    # define the par output
    path_par = os.path.join(path_out, f'out_{h_x:.3f}_{k_l:.3f}_{k_r:.3f}/')
    os.makedirs(path_par, exist_ok=True)

    # initialise the MPS for the indicated chain length
    mps = MPS(L)

    # define the MPO 
    h = MPO_main(h=h_x,k_left=k_l, k_right=k_r, J=J, pol=pol)

    # define the contractions (it needs an mps and a MPO class as imputs)
    cont = CONT(mps=mps,H=h)

    # randomize initial MPS and CONT
    mps.random()
    cont.random()

    # Initialize your dmrg (set low bond dimension to make the system grow faster)
    sys = dmrg(cont=cont,chi=10,cut=1e-12)

    # Grow the system up to the desired dimension
    # En = sys.infinite()

    En = [0.0]
    # open the energy sweep file and write the starting energy
    with open(path_par + 'E_sweep.txt','w') as f:
        f.write(f'{En[-1]} \n')

    # Increase the bond dimension to the desired one 
    sys.chi = chi

    # run the first one and half sweep - do not record!
    for site,dir in mps.first_sweep():
        _,_,_ = sys.step2sites(site,dir=dir)
        
        # write sweep energy
        # with open(path_par + 'E_sweep.txt','a') as f:
        #     f.write(f'{E} \n')

    # Set up counter and energy check
    En_temp = np.zeros(2*L-8)
    En_temp[0] = 1

    # Now we can sweep the system  (ideally until convergence)
    sweep = 0
    while np.abs(En_temp[0] - En_temp[-1]) > 1e-7:
        j = 0
        for site,dir in mps.sweep():
            En_temp[j],S,_ = sys.step2sites(site,dir=dir)
            # write sweep energy
            with open(path_par + 'E_sweep.txt','a') as f:
                f.write(f'{En_temp[j]} \n')
            j +=1
        # increase system bond dimension for more accuracy
        # sys.chi += 100 - adds too much computation time

        if sweep >= 7:
             break
        
        sweep += 1

    # define observables
    obs = observables(mps)

    # Final sweep to store observables
    for site,dir in mps.right_sweep():
        En,S,_ = sys.step2sites(site,dir=dir,stage='Final')

        # Store local magnetization
        with open(path_par + 'X.txt','a') as fz:
                fz.write(f'{site} {obs.single_site(site,h.X).real} \n')
        with open(path_par + 'Y.txt', 'a') as fz:
                fz.write(f'{site} {obs.single_site(site,h.Y).real} \n')
        with open(path_par + 'Z.txt', 'a') as fz:
                fz.write(f'{site} {obs.single_site(site,h.Z).real} \n')

        # Store entanglement entropy
        with open(path_par + 'S.txt','a') as fz:
                fz.write(f'{site} {site+1} {S} \n')

        # Store all two point correlations from site
        obs.all_corr(path_par + 'XX.txt', site, string=h.Id , obs1=h.X)
        obs.all_corr(path_par + 'YY.txt', site, string=h.Id , obs1=h.Y)
        obs.all_corr(path_par + 'ZZ.txt', site, string=h.Id , obs1=h.Z)

        obs.all_corr(path_par + 'XZ.txt', site, string=h.Id , obs1=h.X, obs2=h.Z)
        obs.all_corr(path_par + 'XY.txt', site, string=h.Id , obs1=h.X, obs2=h.Y)
        obs.all_corr(path_par + 'YZ.txt', site, string=h.Id , obs1=h.Y, obs2=h.Z)

        obs.all_corr(path_par + 'XYZ.txt',site, string=h.Y, obs1=h.X, obs2=h.Z)

    end_time = datetime.now()
    diff = end_time - start_time
    total_seconds = int(diff.total_seconds())

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    print(f'Parameter ({h_x:.3f}, {k_l:.3f}, {k_r:.3f}) done !!! Time elapsed: {hours:02}:{minutes:02}:{seconds:02}')

def dmrg_line(L, pol, scan_var, values, opp, set, home, J=1):
    ''' Perform a single line of parameter values for DMRG'''

    folder = f"dmrg_results/L{L}_{opp}{set}_{scan_var}{np.min(values)}-{np.max(values)}"
    if os.path.exists(os.path.join(os.getcwd(), folder)):
        return 1
    # determine h and k
    for ind, val in tqdm(enumerate(values)):
        if scan_var == "h":
            h = val
            k_l = k_r = set
        else:
            h = set     # fixed field, change if needed
            k_l = k_r = val

        # create folder for results
        task_id = f"L{L}_{opp}{set}_{scan_var}{np.min(values)}-{np.max(values)}/{h:.2f}_{k_l:.2f}_{k_r:.2f}"
        workdir = f"run_{task_id}"
        os.makedirs(workdir, exist_ok=True)
        os.chdir(workdir)

        parameter = (h,k_l,k_r)
        dmrg_main(L, parameter, pol, task_id='.', J=J)

        os.chdir("../..")
    
    # go back home
    os.chdir(home)

In [55]:
dmrg_main(L, (1, 0.5, 0.5), '--', '.')

Parameter (1.000, 0.500, 0.500) done !!! Time elapsed: 00:00:15


In [56]:
path = os.path.join(os.getcwd(), "OUT", 'out_1.000_0.500_0.500', "E_sweep.txt")

E_pietro = np.loadtxt(path)[-1]
print(E_pietro)
print(E)
print(1 - (E_pietro/E))
print(np.log10(1 - np.abs(E_pietro/E)))  # 10-12 agreement with one another - good sign

-15.093157006619103
-15.093157006646225
1.7968959653558159e-12
-11.74547706645241


In [58]:
# try 10x check
E_10x, _ = dmrg_sim(10, 10, 5, 5, filename, truth=False)
dmrg_main(L, (10, 5, 5), '--', '.', J=10)
path = os.path.join(os.getcwd(), "OUT", 'out_10.000_5.000_5.000', "E_sweep.txt")
E_pietro10x = np.loadtxt(path)[-1]


INFO    : myModel: reading 'bc_MPS'='finite'
INFO    : myModel: reading 'L'=10
INFO    : myModel: reading 'Jx'=10
INFO    : myModel: reading 'hz'=10
INFO    : myModel: reading 'kl'=5
INFO    : myModel: reading 'kr'=5
INFO    : myModel: reading 'truth'=False
INFO    : TwoSiteDMRGEngine: subconfig 'trunc_params'=Config(<2 options>, 'trunc_params')
INFO    : TwoSiteDMRGEngine: reading 'combine'=True
INFO    : TwoSiteDMRGEngine: reading 'mixer'=True
INFO    : activate DensityMatrixMixer with initial amplitude 1e-05
INFO    : Running sweep with optimization
INFO    : trunc_params: reading 'chi_max'=100
INFO    : trunc_params: reading 'svd_min'=1e-10
INFO    : checkpoint after sweep 1
energy=-150.9315700664623989, max S=0.7579105991419965, age=10, norm_err=9.3e-02
Current memory usage 444.7MB, wall time: 1.8s
Delta E = nan, Delta S = 5.9486e-01 (per sweep)
max trunc_err = 2.3027e-20, max E_trunc = 1.4211e-13
chi: [2, 4, 8, 16, 30, 16, 8, 4, 2]
INFO    : Running sweep with optimization
INFO  

In [63]:
print(E_pietro10x)
print(E_10x)
print(1 - (E_pietro10x/E_10x))
print(np.log10(1 - np.abs(E_pietro10x/E_10x)))  # 10-12 agreement with one another - good sign

print("COMPARING TO PREVIOUS")
print("Order of magnitude of error, Pietro:", np.log10(np.abs(10 - E_pietro10x/E_pietro)))
print("Order of magnitude of error, Mine:", np.log10(np.abs(10 - E_10x/E)))

-150.93157006619154
-150.93157006646248
1.7951196085164156e-12
-11.745906609153586
COMPARING TO PREVIOUS
Order of magnitude of error, Pietro: -13.471716186582249
Order of magnitude of error, Mine: -13.796227278095754
